<h1>Содержание<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Подготовка" data-toc-modified-id="Подготовка-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Подготовка</a></span></li><li><span><a href="#Обучение" data-toc-modified-id="Обучение-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Обучение</a></span></li><li><span><a href="#Выводы" data-toc-modified-id="Выводы-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Выводы</a></span></li><li><span><a href="#Чек-лист-проверки" data-toc-modified-id="Чек-лист-проверки-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Чек-лист проверки</a></span></li></ul></div>

# Проект для «Викишоп»

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию. 

Необходимо обучить модель классифицировать комментарии на позитивные и негативные. Для этого можно использовать набор данных с разметкой о токсичности правок.

**Цель:**

Построить модель со значением метрики качества *F1* не меньше 0.75. 

**Этапы выполнения проекта:**

1. Загрузить и подготовить данные.
2. Обучите модели. 
3. Выводы.

**Описание данных**

Данные находятся в файле `toxic_comments.csv`. Столбец *text* в нём содержит текст комментария, а *toxic* — целевой признак.

In [1]:
!pip install contractions --q

In [2]:
import pandas as pd
import re

import nltk

from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('punkt')

import contractions
import pymystem3

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import f1_score, make_scorer

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from tqdm.notebook import tqdm

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\stude\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\stude\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\stude\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\stude\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
RANDOM_STATE = 12345

## Подготовка

In [4]:
tox_com = pd.read_csv('toxic_comments.csv')
tox_com.head()

,Unnamed: 0,text,toxic
0,0,Explanation\nWhy the edits made under my usern...,0
1,1,D'aww! He matches this background colour I'm s...,0
2,2,"Hey man, I'm really not trying to edit war. It...",0
3,3,"""\nMore\nI can't make any real suggestions on ...",0
4,4,"You, sir, are my hero. Any chance you remember...",0


Удалим первый столбец, который явно отображает индекс строки. Удалим, а не используем его в качестве индекса, так как мало ли в индексации в файле есть ошибки.

In [5]:
tox_com = tox_com[['text', 'toxic']]
tox_com.head()

,text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


В датафрейме есть два столбца: с текстом и с индикатором того, является ли текст токсичным. Необходимо обучить модель машинного обучения, предсказывающую токсичен ли текст.

In [6]:
# общая информация о датафрейме
tox_com.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159292 entries, 0 to 159291
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    159292 non-null  object
 1   toxic   159292 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.4+ MB


Пропущенные значения отсутствуют. Посмотрим действительно ли в столбце ```toxic``` только 0 и 1.

In [7]:
tox_com['toxic'].unique()

array([0, 1], dtype=int64)

Посмотрим верно ли, что 1 означает, что текст токсичен.

In [8]:
tox_com[tox_com['toxic'] == 1]

,text,toxic
6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1
12,Hey... what is it..\n@ | talk .\nWhat is it......,1
16,"Bye! \n\nDon't look, come or think of comming ...",1
42,You are gay or antisemmitian? \n\nArchangel WH...,1
43,"FUCK YOUR FILTHY MOTHER IN THE ASS, DRY!",1
...,...,...
159215,"""\n\n our previous conversation \n\nyou fuckin...",1
159235,YOU ARE A MISCHIEVIOUS PUBIC HAIR,1
159262,Your absurd edits \n\nYour absurd edits on gre...,1
159267,"""\n\nHey listen don't you ever!!!! Delete my e...",1


Видим, что действительно 1 в столбце ```toxic``` означает токсичность комментария.

Теперь лемматизируем текст. Оставим в описаниях только латинские символы и пробелы.

In [9]:
# функция, оставляющая в тексте только латинские символы и пробелы
def clear_text(text):
    expanded_text = contractions.fix(text) # используем библ. contractions для расширения, например, weren't до were not
    return ' '.join(re.sub(r'[^a-zA-Z ]', ' ', expanded_text).split())

In [10]:
# проверка работы функции очистки
display(tox_com['text'][0])
clear_text(tox_com['text'][0])

"Explanation\nWhy the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27"

'Explanation Why the edits made under my username Hardcore Metallica Fan were reverted They were not vandalisms just closure on some GAs after I voted at New York Dolls FAC And please do not remove the template from the talk page since I am retired now'

In [11]:
# вспомогательная функция для лемматизации
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [12]:
# функция лемматизации текста
def lemmatize(text):
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    lemmatized_words_list = [lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in pos_tags]
    lemm_text = " ".join(lemmatized_words_list)
    return lemm_text

In [13]:
# проверка работы функции лемматизации
display((tox_com['text'][0]))
lemmatize(clear_text(tox_com['text'][0]))

"Explanation\nWhy the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27"

'Explanation Why the edits make under my username Hardcore Metallica Fan be revert They be not vandalisms just closure on some GAs after I vote at New York Dolls FAC And please do not remove the template from the talk page since I be retire now'

Теперь создадим столбец ```lemm_text``` с лемматизированным текстом.

In [14]:
tqdm.pandas()
tox_com['lemm_text'] = tox_com['text'].progress_apply(lambda x: lemmatize(clear_text(x)))
tox_com.head()

  0%|          | 0/159292 [00:00<?, ?it/s]

,text,toxic,lemm_text
0,Explanation\nWhy the edits made under my usern...,0,Explanation Why the edits make under my userna...
1,D'aww! He matches this background colour I'm s...,0,D aww He match this background colour I be see...
2,"Hey man, I'm really not trying to edit war. It...",0,Hey man I be really not try to edit war It be ...
3,"""\nMore\nI can't make any real suggestions on ...",0,More I can not make any real suggestion on imp...
4,"You, sir, are my hero. Any chance you remember...",0,You sir be my hero Any chance you remember wha...


В датафрейме появился столбец ```lem_text``` с лемматизированным текстом описаний. Далее обучим несколько моделей машинного обучения для классификации текста пользовательского описания как токсичного или не токсичного.

## Обучение

In [15]:
# разделим датафрейм на признаки и таргет
X = tox_com['lemm_text']
y = tox_com['toxic']
display(X.shape)
y.shape

(159292,)

(159292,)

Разобьем данные на тренировочную и тестовую выборки.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=RANDOM_STATE)
print('train -', X_train.shape, y_train.shape)
print('test -', X_test.shape, y_test.shape)

train - (143362,) (143362,)
test - (15930,) (15930,)


In [17]:
# создадим корпус текстов тренировочной и тестовой выборок
corpus_train = X_train.values
corpus_test = X_test.values

Далее обучим несколько моделей машинного обучения: логистическую регрессию, дерево решений и опорные векторы. Для этого используем пайплайны.

In [18]:
pipe_final = Pipeline([
    ('vect', TfidfVectorizer()),
    ('models', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

In [19]:
param_grid = [
    # словарь для модели DecisionTreeClassifier()
    {
        'models': [DecisionTreeClassifier(random_state=RANDOM_STATE)] 
    },
    
    # словарь для модели LogisticRegression()
    {
        'models': [LogisticRegression(
            random_state=RANDOM_STATE, 
            max_iter=1000
        )],
        'models__C': range(5, 15)
    },
    
    # словарь для модели SVC()
    {
        'models': [LinearSVC(
            random_state=RANDOM_STATE
        )],
        'models__C': range(1, 5)
    }
]

In [20]:
f1_scorer = make_scorer(f1_score)

In [22]:
randomized_search = RandomizedSearchCV(
    pipe_final, 
    param_grid, 
    cv=5,
    scoring=f1_scorer,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

randomized_search.fit(X_train, y_train)

print('Лучшая модель и её параметры:\n\n', randomized_search.best_estimator_)
print ('Средняя метрика лучшей модели на кросс-валидации:', randomized_search.best_score_)

Лучшая модель и её параметры:

 Pipeline(steps=[('vect', TfidfVectorizer()),
                ('models', LinearSVC(C=1, random_state=12345))])
Метрика лучшей модели на кросс-валидации: 0.7858047212824382


Получили в качестве лучшей модели LinearSVC со средней F1-оценкой на кросс-валидации равной 0.786.

Теперь протестируем лучшую модель на тестовых данных.

In [24]:
y_pred = randomized_search.best_estimator_.predict(X_test)

In [25]:
f1_score(y_test, y_pred)

0.7977915804002761

По результатам обучения нескольких моделей в качестве лучшей модели была выбрана LinearSVC со средней F1-оценкой на кросс-валидации равной 0.786. На тестовых данных F1-оценка лучшей модели равна 0.798, что является удовлетворительным результатом, учитывая требования заказчика к F1-оценке о том, что она должна быть равна значению не менее 0.75. 

## Выводы

В качестве входных данных был получен список текстов описаний товаров интернет-магазина "Викишоп" с отметками о том токсичен ли текст. Необходимо было создать инструмент, определющий токсичные комментарии для их последующей модерации. Тексты были очищены от всех символов, кроме латинских и пробелов, и лемматизированы, то есть слова были приведены к своей начальной форме. Данные были разбиты на тренировочную и тестовую выборки.

Затем был создан пайплайн, включающий в себя расчет матрицы с TF-IDF значениями, показывающие важность слов в конкретном тексте относительно всего массива текстов, и дерево решений в качестве модели машинного обучения. Далее был использован инструмент RandomizedSearchCV для поиска лучшей модели с использованием кросс-валидации. Рассмтривались три модели машинного обучения: логистическая регрессия, дерево решений и линейная машина опорных векторов. На валидационных данных лучшая средняя F1-оценка была у линейной машины опорных векторов (LinearSVC). Ее значением округленно равнялось 0.786. Лучшая модель была проверена на тестовых данных. F1-оценка LinearSVC модели на тестовых данных равнялась 0.798.

Таким образом, в качестве модели машинного обучения для определения токсичных описаний рекомендуется использовать модель машинного обучения LinearSVC.

## Чек-лист проверки

- [x]  Jupyter Notebook открыт
- [x]  Весь код выполняется без ошибок
- [x]  Ячейки с кодом расположены в порядке исполнения
- [x]  Данные загружены и подготовлены
- [x]  Модели обучены
- [x]  Значение метрики *F1* не меньше 0.75
- [x]  Выводы написаны